In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

In [36]:
class OLSNormalEquation:

    def __init__(self):
        self.coef_ = None

    @staticmethod
    def has_perfect_multicollinearity(X):
        '''
        to check if design matrix X has perfect multicollinearity. perfect multicollinearity makes XTX singular, so (XTX)-1 does not exist
        '''

        rank = np.linalg.matrix_rank(X)
        n_cols = X.shape[1]
        return rank < n_cols

    def fit(self,X,y):
        '''
        input is design matrix X and response matrix y
        '''
        if self.has_perfect_multicollinearity(X):
            raise ValueError("Perfect multicollinearity detected")

        XtX = X.T @ X
        Xty = X.T @ y
        self.coef_ = np.linalg.inv(XtX) @ Xty
        return self

    def predict(self,X):
        if self.coef_ is None:
            raise ValueError("Model has not been fitted yet")

        return X @ self.coef_




start from the loss func which is the rss bowl
L = (y - y_hat)^2 = (y - b0 - b1x)^2
L = (y - X.T @ b)
L is a func of b -> L(b)

get gradient func del L
b0 = b0 - alpha * del L
b1 = b1 - alpha * del L
at each iteration calculate L
stop when L converges 

In [ ]:
# gradient descent minimises MSE loss

class OLS_GradientDescent:
    def __init__(self):
        self.coef_ = None
        self.loss_history_ = None

    def fit(self, X, y, learning_rate=1e-10, n_iterations=10000):

        self.loss_history_ = []
        
        n_samples, n_features = X.shape
        self.coef_ = np.zeros(n_features)

        for _ in range(n_iterations):
            y_pred = X @ self.coef_
            residuals = (y-y_pred)

            loss = np.mean(residuals ** 2)
            self.loss_history_.append(loss)

            gradient = (-2/n_samples) * X.T @ residuals
            self.coef_ -= learning_rate * gradient

        return self

    def predict(self,X):
        if self.coef_ is None:
           raise ValueError("Model has not been fitted.")
        return X @ self.coef_

    


In [29]:
df = sm.datasets.longley.load_pandas().data


In [55]:
y = df["TOTEMP"].to_numpy()
X = df.drop(columns="TOTEMP").to_numpy()
X = np.column_stack((
    np.ones(len(X)),
    X
))
print(X.shape)
print(y.shape)

(16, 7)
(16,)


In [41]:
norm = OLSNormalEquation()
norm.fit(X,y)
print(norm.coef_)
print(norm.predict(X))

[-3.48225864e+06  1.50618734e+01 -3.58191797e-02 -2.02022981e+00
 -1.03322687e+00 -5.11041047e-02  1.82915147e+03]
[60055.66497512 61216.01894971 60124.71783951 61597.11962768
 62911.29041526 63888.31622261 65153.05396435 63774.18536539
 66004.70023315 67401.61091316 68186.27393591 66552.06005207
 68810.55498044 69649.67631516 68989.0734921  70757.76282856]


In [85]:
GD = OLS_GradientDescent()
GD.fit(X,y)
print(GD.coef_)
print(GD.predict(X))

[nan nan nan nan nan nan nan]
[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]


/var/folders/1p/y7cndqs50bdgrly60h_677pw0000gn/T/ipykernel_21781/803520800.py:19: RuntimeWarning: overflow encountered in square
  loss = np.mean(residuals ** 2)
/var/folders/1p/y7cndqs50bdgrly60h_677pw0000gn/T/ipykernel_21781/803520800.py:23: RuntimeWarning: invalid value encountered in subtract
  self.coef_ -= learning_rate * gradient


In [58]:
model = sm.OLS(y, X)
results = model.fit()

print(results.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     330.3
Date:                Wed, 09 Sep 2026   Prob (F-statistic):           4.98e-10
Time:                        16:02:29   Log-Likelihood:                -109.62
No. Observations:                  16   AIC:                             233.2
Df Residuals:                       9   BIC:                             238.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -3.482e+06    8.9e+05     -3.911      0.0

In [68]:
y_hat = results.predict(X)
y_hat

array([60055.65997022, 61216.01394238, 60124.71283222, 61597.11462191,
       62911.28540922, 63888.31121531, 65153.04895638, 63774.18035685,
       66004.69522738, 67401.60590543, 68186.2689271 , 66552.05504251,
       68810.54997358, 69649.67130802, 68989.06848602, 70757.75782518])